In [ ]:
pip install requests beautifulsoup4 pandas numpy scikit-learn transformers torch streamlit matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
SAVE_PATH   = "diseases_dataset.csv"
FAILED_PATH = "failed_diseases.csv"

In [ ]:
!pip install wikipedia-api --quiet

import os, sys, json, re, time, pickle, warnings
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# Create folder structure
for folder in ["data","data/images","models","figures","app","scraping"]:
    os.makedirs(folder, exist_ok=True)

SAVE_PATH   = "data/diseases_raw.csv"
FAILED_PATH = "data/failed.csv"
IMAGE_DIR   = "data/images"
HEADERS     = {"User-Agent": "DiseaseChecker/1.0 (educational project)"}

import tensorflow as tf
import torch
print(f" TensorFlow: {tf.__version__}")
print(f" PyTorch:    {torch.__version__}")
print(f" GPU: {'T4 GPU ' if tf.config.list_physical_devices('GPU') else 'CPU only '}")
print(" Folders created: data/ | data/images/ | models/ | figures/ | app/ | scraping/")



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 TensorFlow: 2.21.0
 PyTorch:    2.11.0+cpu
 GPU: CPU only 
 Folders created: data/ | data/images/ | models/ | figures/ | app/ | scraping/


In [ ]:
!{sys.executable} -m pip install requests beautifulsoup4 pandas numpy \
    scikit-learn torch torchvision \
    Pillow tqdm matplotlib seaborn --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ── helper: safe HTTP request with retry ────────────────

def safe_request(url, retries=5, delay=3):
    for i in range(retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200: return res
            elif res.status_code == 429:
                print("  Rate limited… waiting 15s")
                time.sleep(15)
        except Exception as e:
            print(f"  Error: {e}")
        time.sleep(delay)
    return None

In [2]:
# ── helper: get Wikipedia summary (title, description, image, url) ──
def get_summary(name):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{quote(name)}"
    res = safe_request(url)
    if not res: return None
    try:
        d = res.json()
        return {
            "title"      : d.get("title", name),
            "description": d.get("extract", ""),
            "image_url"  : d.get("thumbnail", {}).get("source", ""),
            "url"        : d.get("content_urls", {}).get("desktop", {}).get("page", "")
        }
    except: return None

In [3]:
# ── helper: download image locally ──────────────────────
def download_image(url, name):
    if not url: return ""
    try:
        res = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=10)
        if res.status_code != 200: return ""
        path = os.path.join(IMAGE_DIR, name.replace(" ","_").replace("/","_") + ".jpg")
        with open(path,"wb") as f: f.write(res.content)
        return path
    except: return ""

In [4]:
# ── helper: extract symptoms from Wikipedia full page ───
def get_symptoms_from_wikipedia(soup):
    symptoms = []
    # Strategy 1: section header
    for header in soup.find_all(["h2","h3"]):
        if any(w in header.text.lower() for w in ["symptom","sign and","signs and","clinical"]):
            section = header.find_next_sibling()
            while section and section.name not in ["h2","h3"]:
                if section.name == "ul":
                    for li in section.find_all("li", recursive=False):
                        text = re.sub(r"\[.*?\]","",li.get_text(" ",strip=True))
                        text = re.split(r"[;\n]", text)[0].strip().lower()
                        if text and len(text.split()) <= 8:
                            symptoms.append(text)
                    break
                section = section.find_next_sibling()
            if symptoms: return symptoms
    # Strategy 2: infobox
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)
    if infobox:
        for row in infobox.find_all("tr"):
            th = row.find("th"); td = row.find("td")
            if th and td and "symptom" in th.get_text().lower():
                raw = td.get_text(", ", strip=True)
                for item in re.split(r"[,;\n]", raw):
                    item = re.sub(r"\[.*?\]","",item).strip().lower()
                    if item and len(item.split()) <= 8:
                        symptoms.append(item)
                break
    return symptoms



In [5]:
def scrape_disease(name):
    summary = get_summary(name)
    if not summary: return None, "summary_failed"
    page_res = safe_request(summary["url"])
    if not page_res: return None, "page_fetch_failed"
    soup     = BeautifulSoup(page_res.text, "html.parser")
    symptoms = get_symptoms_from_wikipedia(soup)
    if not symptoms:
        keyword_pool = [
            "fever","cough","fatigue","pain","headache","nausea","vomiting",
            "diarrhea","rash","swelling","shortness of breath","chest pain",
            "dizziness","weight loss","weight gain","anxiety","depression",
            "confusion","weakness","tremor","itching","jaundice","chills",
            "sweating","insomnia","memory loss","seizure","paralysis",
            "muscle pain","joint pain","sore throat","runny nose","bloating",
            "palpitations","bleeding","bruising","numbness","tingling",
            "vision loss","hearing loss","loss of appetite","dehydration",
            "pallor","cyanosis","edema","ascites","hematuria"
        ]
        desc_lower = summary["description"][:500].lower()
        symptoms = [kw for kw in keyword_pool if kw in desc_lower]
    if not symptoms: return None, "no_symptoms"
    local_img = download_image(summary["image_url"], summary["title"])
    return {
        "disease_name"    : summary["title"],
        "search_name"     : name,
        "description"     : summary["description"][:500],
        "symptoms"        : " | ".join(symptoms),
        "symptoms_count"  : len(symptoms),
        "image_url"       : summary["image_url"],
        "local_image_path": local_img
    }, None



In [6]:
DISEASES = [
    "Influenza",    "Common cold",    "Pneumonia",    "Tuberculosis",    "Typhoid fever",
    "Cholera",    "Leprosy",    "Diphtheria",    "Pertussis",    "Tetanus",
    "Plague",    "Brucellosis",    "Listeriosis",    "Leptospirosis",    "Melioidosis",
    "Tularemia",    "Anthrax",    "Botulism",    "Salmonellosis",    "Shigellosis",
    "Campylobacteriosis",    "Helicobacter pylori",    "Legionellosis",    "Lyme disease",    "Rocky Mountain spotted fever",
    "Q fever",    "Ehrlichiosis",    "Anaplasmosis",    "Bartonellosis",    "Yersiniosis",
    "Pasteurellosis",    "Nocardiosis",    "Actinomycosis",    "Necrotizing fasciitis",    "Gas gangrene",
    "Staphylococcal infection",    "Streptococcal pharyngitis",    "Scarlet fever",    "Rheumatic fever",    "Impetigo",
    "Erysipelas",    "Cellulitis",    "Osteomyelitis",    "Septic arthritis",    "Endocarditis",
    "Myocarditis",    "Pericarditis",    "Meningitis",    "Brain abscess",    "Spinal epidural abscess",
    "Psoas abscess",    "Liver abscess",    "Splenic abscess",    "Subphrenic abscess",    "COVID-19",
    "HIV/AIDS",    "Hepatitis A",    "Hepatitis B",    "Hepatitis C",    "Hepatitis D",
    "Hepatitis E",    "Chickenpox",    "Measles",    "Mumps",    "Rubella",
    "Dengue fever",    "Malaria",    "Zika virus",    "Ebola virus disease",    "Marburg virus disease",
    "Rabies",    "Yellow fever",    "West Nile fever",    "Chikungunya",    "Monkeypox",
    "Smallpox",    "Herpes simplex",    "Herpes zoster",    "Epstein-Barr virus",    "Cytomegalovirus infection",
    "Norovirus",    "Rotavirus",    "Adenovirus infection",    "Respiratory syncytial virus",    "Human metapneumovirus",
    "Parainfluenza virus",    "Enterovirus",    "Coxsackievirus",    "Echovirus",    "Poliovirus",
    "Hantavirus",    "Nipah virus",    "Crimean-Congo hemorrhagic fever",    "Rift Valley fever",    "Viral hemorrhagic fever",
    "Lymphocytic choriomeningitis",    "Parvovirus B19",    "Molluscum contagiosum",    "Warts",    "Human papillomavirus",
    "Leishmaniasis",    "Trypanosomiasis",    "Chagas disease",    "Toxoplasmosis",    "Cryptosporidiosis",
    "Giardiasis",    "Amebiasis",    "Balantidiasis",    "Cyclosporiasis",    "Microsporidiosis",
    "Babesiosis",    "Schistosomiasis",    "Fascioliasis",    "Clonorchiasis",    "Opisthorchiasis",
    "Paragonimiasis",    "Taeniasis",    "Neurocysticercosis",    "Echinococcosis",    "Trichinosis",
    "Strongyloidiasis",    "Hookworm infection",    "Ascariasis",    "Enterobiasis",    "Trichuriasis",
    "Lymphatic filariasis",    "Loiasis",    "Onchocerciasis",    "Dracunculiasis",    "Toxocariasis",
    "Baylisascariasis",    "Gnathostomiasis",    "Angiostrongyliasis",    "Mansonelliasis",    "Candidiasis",
    "Aspergillosis",    "Cryptococcosis",    "Histoplasmosis",    "Coccidioidomycosis",    "Blastomycosis",
    "Paracoccidioidomycosis",    "Mucormycosis",    "Sporotrichosis",    "Chromoblastomycosis",    "Mycetoma",
    "Dermatophytosis",    "Tinea corporis",    "Tinea pedis",    "Tinea capitis",    "Tinea versicolor",
    "Ringworm",    "Onychomycosis",    "Pneumocystis pneumonia",    "Asthma",    "Chronic obstructive pulmonary disease",
    "Bronchitis",    "Bronchiectasis",    "Emphysema",    "Pulmonary fibrosis",    "Sarcoidosis",
    "Pulmonary hypertension",    "Pulmonary embolism",    "Pleural effusion",    "Pneumothorax",    "Hemothorax",
    "Pleurisy",    "Lung abscess",    "Sinusitis",    "Rhinitis",    "Laryngitis",
    "Tracheitis",    "Epiglottitis",    "Croup",    "Whooping cough",    "Silicosis",
    "Asbestosis",    "Coal workers pneumoconiosis",    "Hypersensitivity pneumonitis",    "Eosinophilic pneumonia",    "Cryptogenic organizing pneumonia",
    "Acute respiratory distress syndrome",    "Obstructive sleep apnea",    "Hypertension",    "Heart failure",    "Coronary artery disease",
    "Myocardial infarction",    "Angina pectoris",    "Atrial fibrillation",    "Ventricular tachycardia",    "Ventricular fibrillation",
    "Heart block",    "Sick sinus syndrome",    "Cardiomyopathy",    "Dilated cardiomyopathy",    "Hypertrophic cardiomyopathy",
    "Restrictive cardiomyopathy",    "Arrhythmogenic cardiomyopathy",    "Cardiac tamponade",    "Aortic dissection",    "Aortic aneurysm",
    "Peripheral artery disease",    "Deep vein thrombosis",    "Varicose veins",    "Thrombophlebitis",    "Raynaud phenomenon",
    "Takayasu arteritis",    "Giant cell arteritis",    "Polyarteritis nodosa",    "Kawasaki disease",    "Infective endocarditis",
    "Rheumatic heart disease",    "Mitral valve prolapse",    "Aortic stenosis",    "Aortic regurgitation",    "Mitral stenosis",
    "Mitral regurgitation",    "Tricuspid regurgitation",    "Pulmonary stenosis",    "Gastroenteritis",    "Peptic ulcer",
    "Gastritis",    "Gastroesophageal reflux disease",    "Esophagitis",    "Barrett esophagus",    "Achalasia",
    "Esophageal varices",    "Mallory-Weiss syndrome",    "Gastroparesis",    "Celiac disease",    "Irritable bowel syndrome",
    "Inflammatory bowel disease",    "Crohn disease",    "Ulcerative colitis",    "Microscopic colitis",    "Diverticulitis",
    "Diverticulosis",    "Appendicitis",    "Intestinal obstruction",    "Volvulus",    "Intussusception",
    "Hirschsprung disease",    "Short bowel syndrome",    "Malabsorption syndrome",    "Lactose intolerance",    "Ischemic colitis",
    "Angiodysplasia",    "Colorectal polyps",    "Hemorrhoids",    "Anal fissure",    "Anal fistula",
    "Perianal abscess",    "Rectal prolapse",    "Fecal incontinence",    "Gallstones",    "Cholecystitis",
    "Cholangitis",    "Primary sclerosing cholangitis",    "Primary biliary cholangitis",    "Pancreatitis",    "Chronic pancreatitis",
    "Pancreatic pseudocyst",    "Autoimmune hepatitis",    "Alcoholic liver disease",    "Nonalcoholic fatty liver disease",    "Liver cirrhosis",
    "Portal hypertension",    "Hepatic encephalopathy",    "Spontaneous bacterial peritonitis",    "Budd-Chiari syndrome",    "Diabetes mellitus",
    "Type 1 diabetes",    "Type 2 diabetes",    "Gestational diabetes",    "Diabetic ketoacidosis",    "Hyperosmolar hyperglycemic state",
    "Hypoglycemia",    "Hypothyroidism",    "Hyperthyroidism",    "Graves disease",    "Hashimoto thyroiditis",
    "Thyroiditis",    "Thyroid nodule",    "Goiter",    "Thyroid storm",    "Hyperparathyroidism",
    "Hypoparathyroidism",    "Addison disease",    "Cushing syndrome",    "Pheochromocytoma",    "Primary hyperaldosteronism",
    "Congenital adrenal hyperplasia",    "Acromegaly",    "Gigantism",    "Dwarfism",    "Diabetes insipidus",
    "Syndrome of inappropriate antidiuretic hormone",    "Hyperprolactinemia",    "Multiple endocrine neoplasia",    "Carcinoid syndrome",    "Metabolic syndrome",
    "Obesity",    "Gout",    "Hyperuricemia",    "Hyperlipidemia",    "Familial hypercholesterolemia",
    "Phenylketonuria",    "Galactosemia",    "Wilson disease",    "Hemochromatosis",    "Porphyria",
    "Fabry disease",    "Gaucher disease",    "Niemann-Pick disease",    "Tay-Sachs disease",    "Mucopolysaccharidosis",
    "Migraine",    "Tension headache",    "Cluster headache",    "Trigeminal neuralgia",    "Stroke",
    "Transient ischemic attack",    "Subarachnoid hemorrhage",    "Intracerebral hemorrhage",    "Subdural hematoma",    "Epidural hematoma",
    "Epilepsy",    "Febrile seizures",    "Status epilepticus",    "Alzheimer disease",    "Vascular dementia",
    "Lewy body dementia",    "Frontotemporal dementia",    "Parkinson disease",    "Multiple system atrophy",    "Progressive supranuclear palsy",
    "Corticobasal degeneration",    "Huntington disease",    "Amyotrophic lateral sclerosis",    "Multiple sclerosis",    "Neuromyelitis optica",
    "Guillain-Barre syndrome",    "Chronic inflammatory demyelinating polyneuropathy",    "Myasthenia gravis",    "Lambert-Eaton syndrome",    "Duchenne muscular dystrophy",
    "Becker muscular dystrophy",    "Facioscapulohumeral muscular dystrophy",    "Myotonic dystrophy",    "Spinal muscular atrophy",    "Charcot-Marie-Tooth disease",
    "Peripheral neuropathy",    "Diabetic neuropathy",    "Carpal tunnel syndrome",    "Tarsal tunnel syndrome",    "Thoracic outlet syndrome",
    "Bell palsy",    "Vestibular neuritis",    "Benign paroxysmal positional vertigo",    "Meniere disease",    "Normal pressure hydrocephalus",
    "Idiopathic intracranial hypertension",    "Narcolepsy",    "Restless legs syndrome",    "Essential tremor",    "Tourette syndrome",
    "Cerebral palsy",    "Spina bifida",    "Hydrocephalus",    "Encephalitis",    "Transverse myelitis",
    "Syringomyelia",    "Myelopathy",    "Depression",    "Major depressive disorder",    "Bipolar disorder",
    "Anxiety disorder",    "Generalized anxiety disorder",    "Panic disorder",    "Social anxiety disorder",    "Post-traumatic stress disorder",
    "Obsessive-compulsive disorder",    "Schizophrenia",    "Schizoaffective disorder",    "Delusional disorder",    "Brief psychotic disorder",
    "Attention deficit hyperactivity disorder",    "Autism spectrum disorder",    "Asperger syndrome",    "Rett syndrome",    "Conduct disorder",
    "Oppositional defiant disorder",    "Separation anxiety disorder",    "Selective mutism",    "Specific phobia",    "Agoraphobia",
    "Anorexia nervosa",    "Bulimia nervosa",    "Binge eating disorder",    "Avoidant restrictive food intake disorder",    "Pica",
    "Rumination disorder",    "Insomnia",    "Hypersomnia",    "Parasomnias",    "Sleepwalking",
    "Sleep terror disorder",    "Alcohol use disorder",    "Opioid use disorder",    "Stimulant use disorder",    "Cannabis use disorder",
    "Gambling disorder",    "Internet gaming disorder",    "Borderline personality disorder",    "Antisocial personality disorder",    "Narcissistic personality disorder",
    "Histrionic personality disorder",    "Paranoid personality disorder",    "Schizoid personality disorder",    "Dissociative identity disorder",    "Depersonalization disorder",
    "Conversion disorder",    "Somatic symptom disorder",    "Illness anxiety disorder",    "Body dysmorphic disorder",    "Trichotillomania",
    "Excoriation disorder",    "Hoarding disorder",    "Intermittent explosive disorder",    "Kleptomania",    "Pyromania",
    "Adjustment disorder",    "Complicated grief",    "Acute stress disorder",    "Osteoarthritis",    "Rheumatoid arthritis",
    "Psoriatic arthritis",    "Ankylosing spondylitis",    "Reactive arthritis",    "Juvenile idiopathic arthritis",    "Systemic lupus erythematosus",
    "Sjogren syndrome",    "Scleroderma",    "Polymyositis",    "Dermatomyositis",    "Mixed connective tissue disease",
    "Fibromyalgia",    "Polymyalgia rheumatica",    "Pseudogout",    "Osteoporosis",    "Osteomalacia",
    "Rickets",    "Paget disease of bone",    "Osteogenesis imperfecta",    "Avascular necrosis",    "Osteonecrosis",
    "Stress fracture",    "Compartment syndrome",    "Rhabdomyolysis",    "Plantar fasciitis",    "Achilles tendinopathy",
    "Rotator cuff tear",    "Frozen shoulder",    "Tennis elbow",    "Golfer elbow",    "De Quervain tenosynovitis",
    "Trigger finger",    "Dupuytren contracture",    "Bunion",    "Flat feet",    "Scoliosis",
    "Kyphosis",    "Lordosis",    "Herniated disc",    "Spinal stenosis",    "Spondylolisthesis",
    "Spondylolysis",    "Facet joint syndrome",    "Sacroiliac joint dysfunction",    "Piriformis syndrome",    "Sciatica",
    "Cervical radiculopathy",    "Lumbar radiculopathy",    "Chronic kidney disease",    "Acute kidney injury",    "Nephrotic syndrome",
    "Nephritic syndrome",    "Glomerulonephritis",    "IgA nephropathy",    "Focal segmental glomerulosclerosis",    "Membranous nephropathy",
    "Minimal change disease",    "Lupus nephritis",    "Diabetic nephropathy",    "Hypertensive nephropathy",    "Polycystic kidney disease",
    "Medullary sponge kidney",    "Renal tubular acidosis",    "Fanconi syndrome",    "Bartter syndrome",    "Gitelman syndrome",
    "Nephrogenic diabetes insipidus",    "Hydronephrosis",    "Kidney stones",    "Urinary tract infection",    "Pyelonephritis",
    "Cystitis",    "Urethritis",    "Bladder outlet obstruction",    "Benign prostatic hyperplasia",    "Prostatitis",
    "Urinary incontinence",    "Overactive bladder",    "Interstitial cystitis",    "Vesicoureteral reflux",    "Neurogenic bladder",
    "Anemia",    "Iron deficiency anemia",    "Vitamin B12 deficiency",    "Folate deficiency anemia",    "Hemolytic anemia",
    "Sickle cell disease",    "Thalassemia",    "Spherocytosis",    "G6PD deficiency",    "Aplastic anemia",
    "Myelodysplastic syndrome",    "Polycythemia vera",    "Essential thrombocythemia",    "Primary myelofibrosis",    "Hemophilia",
    "Von Willebrand disease",    "Thrombocytopenia",    "Immune thrombocytopenia",    "Thrombotic thrombocytopenic purpura",    "Hemolytic uremic syndrome",
    "Disseminated intravascular coagulation",    "Antiphospholipid syndrome",    "Hypercoagulable state",    "Factor V Leiden",    "Protein C deficiency",
    "Protein S deficiency",    "Antithrombin deficiency",    "Lung cancer",    "Breast cancer",    "Colorectal cancer",
    "Prostate cancer",    "Stomach cancer",    "Liver cancer",    "Pancreatic cancer",    "Cervical cancer",
    "Ovarian cancer",    "Uterine cancer",    "Bladder cancer",    "Kidney cancer",    "Thyroid cancer",
    "Skin cancer",    "Melanoma",    "Basal cell carcinoma",    "Squamous cell carcinoma",    "Leukemia",
    "Lymphoma",    "Multiple myeloma",    "Brain tumor",    "Glioblastoma",    "Meningioma",
    "Neuroblastoma",    "Retinoblastoma",    "Osteosarcoma",    "Ewing sarcoma",    "Rhabdomyosarcoma",
    "Liposarcoma",    "Esophageal cancer",    "Head and neck cancer",    "Oral cancer",    "Laryngeal cancer",
    "Nasopharyngeal cancer",    "Testicular cancer",    "Penile cancer",    "Vulvar cancer",    "Vaginal cancer",
    "Anal cancer",    "Small intestine cancer",    "Bile duct cancer",    "Gallbladder cancer",    "Adrenal cortical carcinoma",
    "Carcinoid tumor",    "Gastrointestinal stromal tumor",    "Mesothelioma",    "Thymoma",    "Eczema",
    "Psoriasis",    "Acne",    "Rosacea",    "Seborrheic dermatitis",    "Contact dermatitis",
    "Atopic dermatitis",    "Urticaria",    "Angioedema",    "Erythema multiforme",    "Stevens-Johnson syndrome",
    "Pemphigus vulgaris",    "Bullous pemphigoid",    "Dermatitis herpetiformis",    "Lichen planus",    "Lichen sclerosus",
    "Pityriasis rosea",    "Vitiligo",    "Alopecia areata",    "Androgenetic alopecia",    "Telogen effluvium",
    "Folliculitis",    "Furuncle",    "Carbuncle",    "Hidradenitis suppurativa",    "Pilonidal cyst",
    "Keloid",    "Hypertrophic scar",    "Stretch marks",    "Ichthyosis",    "Epidermolysis bullosa",
    "Scabies",    "Pediculosis",    "Bedbug infestation",    "Cutaneous larva migrans",    "Conjunctivitis",
    "Keratitis",    "Uveitis",    "Glaucoma",    "Cataracts",    "Macular degeneration",
    "Diabetic retinopathy",    "Retinal detachment",    "Retinitis pigmentosa",    "Optic neuritis",    "Optic atrophy",
    "Strabismus",    "Amblyopia",    "Ptosis",    "Blepharitis",    "Chalazion",
    "Hordeolum",    "Dacryocystitis",    "Orbital cellulitis",    "Dry eye syndrome",    "Corneal ulcer",
    "Scleritis",    "Episcleritis",    "Endophthalmitis",    "Color blindness",    "Night blindness",
    "Otitis media",    "Otitis externa",    "Mastoiditis",    "Cholesteatoma",    "Otosclerosis",
    "Tinnitus",    "Sensorineural hearing loss",    "Conductive hearing loss",    "Acoustic neuroma",    "Labyrinthitis",
    "Epistaxis",    "Nasal polyps",    "Deviated nasal septum",    "Adenoiditis",    "Tonsillitis",
    "Peritonsillar abscess",    "Retropharyngeal abscess",    "Vocal cord paralysis",    "Hoarseness",    "Dysphonia",
    "Globus sensation",    "Temporomandibular joint disorder",    "Central sleep apnea",    "Dysmenorrhea",    "Amenorrhea",
    "Menorrhagia",    "Premenstrual syndrome",    "Premenstrual dysphoric disorder",    "Polycystic ovary syndrome",    "Endometriosis",
    "Uterine fibroids",    "Uterine polyps",    "Cervicitis",    "Pelvic inflammatory disease",    "Ovarian cyst",
    "Ovarian torsion",    "Ectopic pregnancy",    "Miscarriage",    "Preeclampsia",    "Eclampsia",
    "Gestational hypertension",    "Placenta previa",    "Placental abruption",    "Preterm labor",    "Premature rupture of membranes",
    "Postpartum hemorrhage",    "Postpartum depression",    "Mastitis",    "Breast abscess",    "Galactorrhea",
    "Vaginal discharge",    "Bacterial vaginosis",    "Vulvovaginal candidiasis",    "Trichomoniasis",    "Genital herpes",
    "Genital warts",    "Syphilis",    "Gonorrhea",    "Chlamydia",    "Pelvic organ prolapse",
    "Stress urinary incontinence",    "Neonatal jaundice",    "Neonatal sepsis",    "Respiratory distress syndrome",    "Necrotizing enterocolitis",
    "Bronchiolitis",    "Henoch-Schonlein purpura",    "Pyloric stenosis",    "Imperforate anus",    "Tracheoesophageal fistula",
    "Patent ductus arteriosus",    "Ventricular septal defect",    "Atrial septal defect",    "Tetralogy of Fallot",    "Transposition of great arteries",
    "Coarctation of aorta",    "Hypoplastic left heart syndrome",    "Down syndrome",    "Turner syndrome",    "Klinefelter syndrome",
    "Fragile X syndrome",    "Prader-Willi syndrome",    "Angelman syndrome",    "DiGeorge syndrome",    "Williams syndrome",
    "Marfan syndrome",    "Ehlers-Danlos syndrome",    "Neurofibromatosis",    "Tuberous sclerosis",    "Maple syrup urine disease",
    "Organic acidemia",    "Urea cycle disorders",    "Glycogen storage disease",    "Mitochondrial disease",    "Allergic rhinitis",
    "Allergic asthma",    "Food allergy",    "Drug allergy",    "Latex allergy",    "Insect sting allergy",
    "Anaphylaxis",    "Hereditary angioedema",    "Common variable immunodeficiency",    "Selective IgA deficiency",    "X-linked agammaglobulinemia",
    "Severe combined immunodeficiency",    "Wiskott-Aldrich syndrome",    "Chronic granulomatous disease",    "Complement deficiency",    "Autoimmune polyendocrinopathy",
    "Immune dysregulation syndrome",    "Dental caries",    "Periodontal disease",    "Gingivitis",    "Periodontitis",
    "Dental abscess",    "Pulpitis",    "Apical periodontitis",    "Dry socket",    "Oral candidiasis",
    "Oral lichen planus",    "Leukoplakia",    "Erythroplakia",    "Aphthous stomatitis",    "Angular cheilitis",
    "Glossitis",    "Geographic tongue",    "Burning mouth syndrome",    "Xerostomia",    "Sialadenitis",
    "Salivary gland calculi",    "Bruxism",    "Malocclusion",    "Occupational asthma",    "Byssinosis",
    "Lead poisoning",    "Mercury poisoning",    "Arsenic poisoning",    "Carbon monoxide poisoning",    "Pesticide poisoning",
    "Heat stroke",    "Heat exhaustion",    "Hypothermia",    "Frostbite",    "Decompression sickness",
    "High altitude sickness",    "Radiation sickness",    "Noise-induced hearing loss",    "Repetitive strain injury",    "Cystic fibrosis",
    "Alpha-1 antitrypsin deficiency",    "Familial Mediterranean fever",    "Hereditary hemorrhagic telangiectasia",    "Von Hippel-Lindau disease",    "Li-Fraumeni syndrome",
    "BRCA mutation",    "Lynch syndrome",    "Familial adenomatous polyposis",    "Achondroplasia",    "Albinism",
    "Xeroderma pigmentosum",    "Fanconi anemia",    "Diamond-Blackfan anemia",    "Shwachman-Diamond syndrome",    "Dyskeratosis congenita",
    "Periodic fever syndromes",    "Autoinflammatory diseases"
]



In [ ]:
# ── Run Scraper ──────────────────────────────────────────

import os
import time
import pandas as pd
from urllib.parse import quote

# Files
SAVE_PATH   = "diseases_dataset.csv"
FAILED_PATH = "failed_diseases.csv"
# Loads existing data first (safe to re-run if interrupted)
if os.path.exists(SAVE_PATH):
    existing = pd.read_csv(SAVE_PATH)
    all_data = existing.to_dict("records")
    done_names = set(existing["search_name"].str.lower())
    print(f"Resuming — already have {len(all_data)} diseases")
else:
    all_data, done_names = [], set()

failed = []
todo   = [d for d in DISEASES if d.lower() not in done_names]
print(f"Remaining: {len(todo)} diseases to scrape")

for i, disease in enumerate(todo):
    print(f"[{i+1}/{len(todo)}] {disease}...", end=" ")
    result, error = scrape_disease(disease)
    if result:
        all_data.append(result)
        print(f"✓ ({result['symptoms_count']} symptoms)")
    else:
        failed.append({"disease": disease, "error": error})
        print(f"✗ {error}")
    # Save incrementally
    pd.DataFrame(all_data).to_csv(SAVE_PATH,   index=False)
    pd.DataFrame(failed  ).to_csv(FAILED_PATH, index=False)
    time.sleep(1.5)

df_raw = pd.DataFrame(all_data)
print(f"\n Done! Collected: {len(all_data)} / {len(DISEASES)}")
print(f"   Failed: {len(failed)}")
df_raw.head(3)


Remaining: 817 diseases to scrape
[1/817] Influenza...   Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
✗ summary_failed
[2/817] Common cold...   Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
✗ summary_failed
[3/817] Pneumonia...   Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
✗ summary_failed
[4/817] Tuberculosis...   Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
  Error: name 'requests' is not defined
✗ summary_failed
[5/817] T